# CareFlow — AI Care Coordination Assistant

**IIT Hyderabad Applied AI Professional Certification · Capstone**

This notebook is one milestone in the eight-milestone CareFlow implementation.
All patient, insurance, appointment and referral examples are synthetic. No real PHI is used.

> **Data boundary:** the official CareFlow Data Toolkit assets are the source of truth.
> Intake records feed M1; mock API tables feed M2/M6; the policy corpus feeds M3/M4;
> the golden evaluation set is finalized in M8 and should be started at M4.

## M2 — Tool Agent over CareFlow Mock APIs

### Goal
Give the assistant deterministic tools for reading operational state rather than asking the LLM to invent database facts.

**Primary asset:** `data/careflow/mock_api/`

Expected tables: `plans`, `patients`, `appointments`, `referrals`.

In [1]:
import sys
print(sys.version)
print(sys.executable)

3.14.7 (main, Aug  5 2026, 10:29:49) [Clang 21.0.0 (clang-2100.1.1.101)]
/Users/sgunaseelan/Projects/API/AI-IIT/applied-ai-capstone-starter/.venv-py314/bin/python


In [2]:
from pathlib import Path
import pandas as pd
import json

BASE = Path("data/careflow/mock_api")

def load_table(name):
    for suffix in [".csv", ".json"]:
        p = BASE / f"{name}{suffix}"
        if p.exists():
            return pd.read_csv(p) if suffix == ".csv" else pd.DataFrame(json.loads(p.read_text()))
    return None

tables = {n: load_table(n) for n in ["plans","patients","appointments","referrals"]}
{k: (None if v is None else v.shape) for k,v in tables.items()}

{'plans': None, 'patients': None, 'appointments': None, 'referrals': None}

In [3]:
# Offline fallback so the notebook remains demonstrable.
if any(v is None for v in tables.values()):
    tables = {
        "plans": pd.DataFrame(
            [
                ["MERIDIAN-GOLD", 500, 0.20, 25000, 25, 10],
                ["MERIDIAN-SILVER", 1500, 0.30, 15000, 45, 18],
                ["MERIDIAN-BRONZE", 3000, 0.40, 8000, 45, 18],
                ["CIVIC-BASE", 2000, 0.35, 10000, 45, 18],
            ],
            columns=[
                "plan_code",
                "annual_deductible_usd",
                "coinsurance_rate",
                "out_of_pocket_max_usd",
                "specialist_copay_usd",
                "telehealth_copay_usd",
            ],
        ),
        "patients": pd.DataFrame(
            [
                ["MHP-P-00001", "MERIDIAN-GOLD", "active", 250, "MHP-SITE-01"],
                ["MHP-P-00002", "CIVIC-BASE", "active", 900, "MHP-SITE-02"],
            ],
            columns=[
                "patient_ref",
                "plan_code",
                "eligibility_status",
                "deductible_met_usd",
                "primary_site",
            ],
        ),
        "appointments": pd.DataFrame(
            [
                ["MHP-A-000001", "MHP-P-00001", "Cardiology", "MHP-PR-001", "2026-10-15T10:00", "scheduled", 30],
                ["MHP-A-000002", "MHP-P-00002", "Dermatology", "MHP-PR-002", "2026-10-18T14:00", "scheduled", 30],
            ],
            columns=[
                "appointment_id",
                "patient_ref",
                "specialty",
                "provider_id",
                "scheduled_for",
                "status",
                "duration_minutes",
            ],
        ),
        "referrals": pd.DataFrame(
            [
                ["MHP-R-00001", "MHP-P-00001", "Primary Care", "Cardiology", "Routine", "open", False],
                ["MHP-R-00002", "MHP-P-00002", "Primary Care", "Dermatology", "Priority", "accepted", True],
            ],
            columns=[
                "referral_id",
                "patient_ref",
                "from_specialty",
                "to_specialty",
                "urgency",
                "status",
                "preauth_required",
            ],
        ),
    }

### Tool contracts

In [4]:
def get_patient(patient_ref):
    df = tables["patients"]
    rows = df[df.patient_ref == patient_ref]
    return rows.to_dict("records")

def get_plan(plan_code):
    df = tables["plans"]
    rows = df[df.plan_code == plan_code]
    return rows.to_dict("records")

def get_appointments(patient_ref, specialty=None):
    df = tables["appointments"]
    rows = df[df.patient_ref == patient_ref]
    if specialty:
        rows = rows[rows.specialty.str.lower() == specialty.lower()]
    return rows.to_dict("records")

def get_referrals(patient_ref):
    df = tables["referrals"]
    return df[df.patient_ref == patient_ref].to_dict("records")

TOOLS = {
    "get_patient": get_patient,
    "get_plan": get_plan,
    "get_appointments": get_appointments,
    "get_referrals": get_referrals,
}

In [5]:
# A deterministic tool-agent loop: classify the request, select a tool, execute it.
def route_tool(question):
    q = question.lower()
    if "appointment" in q:
        return "get_appointments"
    if "referral" in q:
        return "get_referrals"
    if "plan" in q or "deductible" in q or "copay" in q:
        return "get_patient"
    return "get_patient"

question = "Show the appointments for MHP-P-00001."
tool_name = route_tool(question)
print("Selected tool:", tool_name)
print(TOOLS[tool_name]("MHP-P-00001"))

Selected tool: get_appointments
[{'appointment_id': 'MHP-A-000001', 'patient_ref': 'MHP-P-00001', 'specialty': 'Cardiology', 'provider_id': 'MHP-PR-001', 'scheduled_for': '2026-10-15T10:00', 'status': 'scheduled', 'duration_minutes': 30}]


### M2 principle

The LLM can decide **which tool to call**, but the tool is the source of truth for relational state.

This avoids hard-coded answers and creates the same tool contracts that can later be exposed through MCP in M6.